In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Pooling

The layers of the previous sections answer local questions. The edge
detector of that section reports, position by position, whether an
edge lies there. A classifier must answer a question about the whole image:
does this Fashion-MNIST picture show a shirt or a sneaker? This section asks
how a network gets from many local answers to one global answer. Two things
are needed.

The first is reach: some unit near the top of the network must have a
receptive field (that section) that covers the whole image. At
full resolution this takes patience, namely fourteen $3\times3$ layers for a
$28\times28$ Fashion-MNIST image. We already know the remedy, which is to
reduce the resolution on the way up (that section). With two
halvings, six such layers are enough.

The second is new. The edge detector responds at the pixel where the edge
lies. Shift the image by one pixel, as happens whenever the object or the
camera moves slightly, and the response moves by one pixel too. A
classifier's answer should not change because of such a shift.

A *pooling layer* halves the resolution as a stride does, but it also decides
what to keep. It replaces each small window of its input by one summary
value, the maximum or the average. The maximum of a window does not change
when a response moves within it, and this is what bears on the second
problem. How much it helps is something we can measure, and the answer is
less than the word "invariance", often used of pooling, suggests. Invariance
proper takes a window as large as the feature map, and even then the border
has a say.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
from functools import partial

## Maximum Pooling and Average Pooling

A pooling operator slides a window of fixed shape, the *pooling window*, over
its input, like the kernel of a convolution. At each position it computes one
output from the entries under the window. Unlike a convolution, it has no
kernel and therefore nothing to learn: *maximum pooling* (*max-pooling* for
short) outputs the largest entry in the window, and *average pooling* outputs
their mean.

![Max-pooling with a pooling window shape of $2\times 2$. The shaded portions are the first output element as well as the input tensor elements used for the output computation: $\max(0, 1, 3, 4)=4$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/pooling.svg)

In the figure a $2\times2$ window slides over a $3\times3$ input
with stride 1, from the upper left, left to right and top to bottom. It fits
at two positions along each axis, so the output is $2\times2$, and each output
entry is the maximum of the four entries under the window:

$$
\begin{aligned}
\max(0, 1, 3, 4) &= 4,\\
\max(1, 2, 4, 5) &= 5,\\
\max(3, 4, 6, 7) &= 7,\\
\max(4, 5, 7, 8) &= 8.
\end{aligned}
$$

Note that five of the nine entries, namely 0, 1, 2, 3, and 6, are the
largest in no window. Small changes to them leave the output unchanged: a
maximum takes no notice of the entries that lose. A pooling layer with a
$k_\textrm{h} \times k_\textrm{w}$ window works the same way.

The function `pool2d` implements the forward computation. It has the loop
structure of `corr2d` in that section, with the multiply-and-sum
over the window replaced by a maximum or a mean. The argument `stride` moves
the window by more than one position at a time; we need it below.

In [ ]:
def pool2d(X, pool_size, mode='max', stride=1):
    k_h, k_w = pool_size
    Y = jnp.zeros(((X.shape[0] - k_h) // stride + 1,
                   (X.shape[1] - k_w) // stride + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            window = X[i * stride: i * stride + k_h,
                       j * stride: j * stride + k_w]
            Y = Y.at[i, j].set(window.max() if mode == 'max'
                               else window.mean())
    return Y

Applied to the input of the figure, `pool2d` should return
4, 5, 7, and 8.

In [ ]:
X = d2l.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

Average pooling of the same input returns the four window means, 2, 3, 5,
and 6.

In [ ]:
pool2d(X, (2, 2), 'avg')

## Tolerance to Small Shifts

How much of a shift does pooling absorb? Let us try the simplest case there
is. Pooling with a window of $2\times2$ and stride 2 splits its input into
disjoint $2\times2$ blocks and reduces each block to one number. Consider a
feature map with a single response of value 1, such as the output of an edge
detector at an isolated edge, and move the response by one pixel at a time.
We compute the max-pooled and the average-pooled outputs for three positions
of the response in row 1 of a $4\times4$ map.

In [ ]:
def single_response(i, j, n=4):
    """An n x n feature map that is 1 at (i, j) and 0 elsewhere."""
    return d2l.tensor([[float(r == i and c == j) for c in range(n)]
                       for r in range(n)])

for j in range(3):
    Z = single_response(1, j)
    print(f'response at (1, {j}): max',
          d2l.numpy(pool2d(Z, (2, 2), stride=2)).tolist(),
          'avg', d2l.numpy(pool2d(Z, (2, 2), 'avg', stride=2)).tolist())

Moving the response from column 0 to column 1 changes neither output: both
positions lie in the same block. Moving it on to column 2 crosses into the
next block, and the output moves with it. A one-pixel shift of a single
response is therefore absorbed when the response stays inside its block,
which happens for half of all horizontal positions, and it moves the pooled
output by one position otherwise. Pooling makes a representation *tolerant*
to some small shifts. It does not make it invariant to them.
the figure draws the max-pooled outputs for all four
positions of the response.

![A single response moved through the four columns of a $4\times4$ feature map (top), its $2\times2$ max-pooled map with stride 2 (middle), and its global maximum (bottom). Heavy lines mark the pooling blocks. The pooled response stays put while the response stays inside its block and jumps when it crosses into the next one; the global maximum never moves.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/pooling-shift.svg)

The two kinds of pooling differ in what they report. Max-pooling reports
the response at full strength, 1, wherever it falls in the block: it answers
whether the feature is present somewhere in the window. Average pooling
reports 0.25, because the response covers one of the four positions: it
measures how much of the window the feature fills, so a strong but isolated
response is diluted by its neighbors.

A stride of 2 keeps one value per block, so the output can jump under a
one-pixel shift, as at the block boundary above. In the extreme case, keeping
every second pixel of a pattern that alternates from pixel to pixel,
$1, 0, 1, 0, \ldots$, gives all ones or all zeros, depending on a shift by
one pixel. This error, which arises when a signal changes faster than the
samples can follow, is called *aliasing*. It affects every stride-2
downsampler, pooled or convolutional. Smoothing the input before subsampling
reduces it [@zhang2019making], and an exercise below works it out.

Tolerance to shifts is what this operation was invented for. The
Neocognitron [@Fukushima.1980] followed each layer of feature detectors
by a layer of cells that responded when a detector fired anywhere in a small
neighborhood. An early form of max-pooling was used in speech recognition
[@Yamaguchi.Sakamoto.Akabane.ea.1990], and the operation was later
proposed as a model of how the visual cortex aggregates evidence for object
recognition [@Riesenhuber.Poggio.1999].

## Global Pooling

What, then, about the invariance that that section asked of the
final answer, whether Waldo is anywhere in the scene? A window of fixed size
cannot deliver it, as we have just seen. A window that covers the whole
feature map can: the maximum over all positions, or the mean, is the same
wherever in the map the response lies (bottom row of
the figure). This is *global pooling*, and it reduces each
channel to one number.

For a single response on a bare map this is true by definition, and no
experiment is needed. In a network, however, global pooling summarizes the
output of a convolution, and a convolution follows a shift only away from
the border (that section). To see what the border does, we move
the single response through all four columns once more and apply a
$3\times3$ kernel of ones with one pixel of zero padding, which copies the
response into every output whose window contains it. Then we pool the result
globally.

In [ ]:
pad = lambda Z: jnp.pad(Z, 1)  # One pixel of zeros on every side

In [ ]:
for j in range(4):
    H = d2l.numpy(d2l.corr2d(pad(single_response(1, j)), d2l.ones((3, 3))))
    print(f'response at (1, {j}): global max {H.max():.2f}, '
          f'global mean {H.mean():.4f}')

The global maximum is 1 wherever the response lies. The global mean,
however, is not constant. Nine windows contain a response in column 1 or 2,
but only six contain one in the border columns 0 and 3, which is the count
of the figure, and the mean falls from $9/16$ to $6/16$. The
conclusion holds beyond this small map: convolutions followed by global
pooling are invariant to shifts that keep the content away from the border,
and only approximately invariant to the others.

In the networks of that section the two tasks of pooling part
ways. Halving the resolution is often left to a convolution with stride 2,
which learns how to combine the entries of a window instead of fixing the
combination to a maximum or a mean, and the network usually ends in *global
average pooling* [@Lin.Chen.Yan.2013]. LeNet, in the next section, does
neither.

## Padding and Stride

Here there is little to learn. The pooling layers of the frameworks take a
window shape, a stride, and a padding, with the meaning they have for
convolutions, and the output shape follows the same rule
(that section): with $p$ rows of padding on each side and stride
$s$, a window of height $k$ on an input of height $n$ gives
$\lfloor(n+2p-k)/s\rfloor+1$ rows, and likewise for the columns. Only one
thing is new, the value that fills the padding. The frameworks expect
four-dimensional input, so we put the $4\times4$ matrix of the numbers 0 to
15 into a tensor with one example and one channel.

In [ ]:
# (example, height, width, channel), as in the previous sections
X = d2l.reshape(d2l.arange(16, dtype=d2l.float32), (1, 4, 4, 1))
X

Pooling summarizes a region, so the usual stride equals the window shape:
the windows then tile the input without overlapping. A $3\times3$ window on
the $4\times4$ input fits once, and the output is the maximum of the
top-left $3\times3$ block, 10. With no parameters, the layer needs no
initialization either.

In [ ]:
# The stride of nnx.max_pool defaults to 1, so we set it to the window shape
nnx.max_pool(X, window_shape=(3, 3), strides=(3, 3))

Overlapping windows need an explicit stride. With padding 1 and stride 2,
the output is $\lfloor(4+2-3)/2\rfloor+1 = 2$ entries high and wide.

In [ ]:
# One (before, after) pair of padding for each spatial axis
pool = partial(nnx.max_pool, window_shape=(3, 3), strides=(2, 2),
               padding=((1, 1), (1, 1)))
pool(X)

Which value fills the padding? A convolution pads with zeros, which add
nothing to a weighted sum. For a maximum, the neutral value is $-\infty$:
padded positions then never win, and each output is the maximum of the real
entries under its window. Here the windows cover the real entries in rows
and columns $\{0, 1\}$ and $\{1, 2, 3\}$, so the outputs are 5, 7, 13, and
15. Padding with $-\infty$ also keeps a property that every maximum has:
adding a constant to every input entry adds the same constant to every
output. We check it on `X - 16`, whose entries all lie below zero.

In [ ]:
pool(X - 16)

Each output is 16 below the previous one, as it should be. Zero padding
would have produced 0 in the three windows that touch the padding, since 0
exceeds every real entry there.

Average pooling is less fortunate: it has no neutral padding value. If the
padded positions count as zeros in the mean, they pull the outputs at the
border toward 0; the alternative is to average only the real entries under
each window.
<!-- d2l:prose id=pooling-md-12 fw=mxnet, pytorch, jax -->

Average pooling here counts the padded zeros by default
(`count_include_pad=True`); `count_include_pad=False` averages only the real
entries.

## Multiple Channels

A convolutional layer sums over its input channels
(that section): every output channel reads all of them. A pooling
layer does nothing of the kind. It pools each input channel separately, so
its output has as many channels as its input. To check this, we concatenate
`X` and `X + 1` along the channel axis to obtain an input with two channels.

In [ ]:
X = d2l.concat([X, X + 1], 3)  # the channel axis is last
X

The pooling layer with padding 1 and stride 2 from above returns two
channels: the first holds the outputs computed earlier, 5, 7, 13, and 15,
and the second the same values plus 1.

In [ ]:
pool(X)

A pooling layer finds nothing by itself. It has no weights, and whatever it
summarizes, a convolution before it had to detect. The parts are now all on
the table: a layer that detects, settings for its shape and reach, channels,
and a summary of neighborhoods. What remains is to see whether, put
together and trained, they classify images better than the MLP did.

## Summary

Pooling replaces each window of its input by its maximum or its mean. It has
no parameters, it pools each channel separately, and its output shape follows
the padding and stride rules of convolutions; max-pooling pads with
$-\infty$, so padding never changes a maximum. A $2\times2$ window with
stride 2 halves the resolution, and it absorbs a shift of a response that
stays inside its window, but not one that crosses into the next. Max-pooling
keeps the strength of the strongest response; average pooling dilutes it
over the window. Only a window as large as the feature map, global pooling,
removes the position of a response altogether, and after a padded
convolution it does so only up to the border.

## Exercises

1. [code] **Average pooling as convolution.** Express
   $k_\textrm{h} \times k_\textrm{w}$ average pooling with stride
   $(s_\textrm{h}, s_\textrm{w})$ on an input with $c$ channels as a
   convolution with $c$ output channels. Give the kernel
   $\mathbf{K}_{o,c}$ for $o = c$ and for $o \neq c$, so that each channel
   is pooled separately, as pooling layers do. On the input
   of the figure, verify that your convolution reproduces
   `pool2d(X, (2, 2), 'avg')`.

1. **Receptive field and cost of downsampling.** Consider $3\times3$
   convolutions with stride 1 and padding 1, with 64 input and 64 output
   channels, on a $224\times224$ input.
    1. How many such layers does a stack need before the receptive field
       of a unit at its top covers the whole image?
    1. Now insert $2\times2$ max-pooling with stride 2 after every second
       convolutional layer. How many convolutional layers are needed now?
    1. Count the multiply-adds of the two stacks, and the comparisons that
       the pooling layers perform. Which part of the cost does pooling
       remove, and what does it cost itself?

1. **Pooling alternatives.**
    1. Do we need a separate minimum-pooling layer, or can it be expressed
       through an operation already in this section?
    1. *Softmax pooling* with temperature $\tau > 0$ replaces the window
       entries $x_1, \ldots, x_m$ by their softmax-weighted average
       $$\sum_{i=1}^m x_i \frac{\exp(x_i / \tau)}{\sum_{j=1}^m \exp(x_j / \tau)}.$$
       Find its limits as $\tau \to 0$ and as $\tau \to \infty$. Give one
       cost that softmax pooling adds relative to max-pooling.

1. **Aliasing and blur-pool.** Naive stride-2 downsampling keeps
   every second entry of its input. Measure frequency in cycles per input
   sample, so that the one-dimensional signals $(1, 0, 1, 0, 1, 0)$ and
   $(0, 1, 0, 1, 0, 1)$ both have frequency $\tfrac{1}{2}$.
    1. Apply naive stride-2 downsampling to both signals, which differ by
       a shift of one position, and compare the two outputs. Why is this
       behavior called *aliasing*?
    1. Which input frequencies can a stride-2 subsampler represent
       faithfully?
    1. Blur-pool [@zhang2019making] counteracts aliasing by applying
       a small averaging filter before subsampling. Work out what
       blur-pool with a two-tap box filter
       $(\tfrac{1}{2}, \tfrac{1}{2})$ computes on the two signals above.
       Which pooling operation from this section does it coincide with?

1. [code] **Backward pass of max-pooling.** Derive the gradient of
   two-dimensional max-pooling with respect to its input, for a
   $k_\textrm{h} \times k_\textrm{w}$ window and stride equal to the
   window size, and implement it without automatic differentiation.
   Verify against the gradient that automatic differentiation computes
   through the built-in max-pooling operation of this section on a small
   random tensor, to a relative error below $10^{-4}$.

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2024/assignment2/),
    ConvolutionalNetworks, naive max-pooling backward pass.*

1. **Max-pooling via ReLU.** ● A convolution is linear, so it cannot
   compute a maximum by itself.
    1. Express $\max(a, b)$ using only affine operations and
       $\textrm{ReLU}(x) = \max(0, x)$.
    1. Use this to implement max-pooling as a stack of convolutions and
       ReLU layers.
    1. Count the channels per layer and the ReLU layers that your
       construction uses for a $2 \times 2$ window and for a
       $3 \times 3$ window. How does the number of ReLU layers grow with
       the number of entries in the window?

[Discussions](https://d2l.discourse.group/t/17999)